In [ ]:
import os

# Safely shift the working directory up to the project root
# This avoids sys.path modifications and allows clean 'src.' imports
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import sqlite3
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Clean package imports from the src/ directory
from src.data_loader import load_and_merge_data
from src.preprocessor import DataPreprocessor
from src.walmart_dataset import WalmartDataset
from src.sales_predictor import SalesPredictor

print("Package imports successful!")

Package imports successful!


In [ ]:
# Load the raw merged data frame.
df = load_and_merge_data('../data')

# Split the training set into a train/test set using pandas (80% train, 20% test).
train_df = df.sampe(frac=0.8, random_state=23)
# The test set is the original data frame minus the training set.
test_df = df.drop(train_df.index)

# Pre-process and scale features
preprocessor = DataPreprocessor()
x_train, y_train = preprocessor.fit_transform(train_df)
x_test, y_test = preprocessor.transform(test_df)

# Wrap the data in a PyTorch Dataset and Dataloader for batch training
train_set = WalmartDataset(x_train, y_train)
test_set = WalmartDataset(x_test, y_test)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

# Initialize the model
model = SalesPredictor(x_train.shape[1])

# Define the loss function and create the optimizer
criterion = nn.L1Loss() # Chosen for interpretability in the context of sales
optimizer = optim.Adam(model.parameters(), lr=0.001) # For simplicity, using Adam as the optimizer.

# Training loop
epochs = 10
model.train() # Set the model to training mode
for epoch in range(epochs):
    epoch_loss = 0.0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Reset gradients
        optimizer.zero_grad()

        # Forward pass through the model
        predictions = model(inputs)

        # Compute the loss
        loss = criterion(predictions, targets)

        # Backward pass (to compute the gradients)
        loss.backward()

        # Optimizer step (to update the model parameters)
        optimizer.step()

        # Accumulate the batch loss
        epoch_loss += loss.item()

    # Calculate the average loss for the epoch
    epoch_loss /= len(train_loader)
    print(f"Epoch {epoch+1}/epochs | Train mean average error (MAE): {epoch_loss:.2f}")